# SpaceX Falcon 9 First-Stage Landing Prediction
**Applied Data Science Capstone — Python Notebook**  
Prepared for the IBM Data Science Capstone assessment  

**Research question:** Can historic launch characteristics predict whether a Falcon 9 first stage lands successfully?  
**Reproducibility:** Run cells in order in a Jupyter environment with internet access. The notebook fetches public data; no model scores or observed counts are pre-invented. The companion PDF is a presentation narrative.  

**Important:** The grading screen specifically requires a **PDF report**, while this notebook supplies the reproducible code. Upload the PDF to the shown field, and retain or share this notebook wherever source code is requested.

## 1. Executive summary and business case
Reusable first stages can materially affect launch economics. We analyze historically recorded launch outcomes and construct baseline classification models. Historical associations are not causal, and the dataset is small and dated; predictions should not be treated as flight-safety judgments.

In [ ]:
# Run once if your environment lacks any dependencies:
# %pip install -q pandas numpy requests beautifulsoup4 lxml matplotlib seaborn scikit-learn folium plotly dash
import io, sqlite3, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from IPython.display import display, HTML
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
sns.set_theme(style='whitegrid')
SEED = 42
BASE = 'https://raw.githubusercontent.com/elgels/IBM-Data-Science-Capstone/main/data/'
def load_csv(filename):
    try:
        return pd.read_csv(filename)
    except FileNotFoundError:
        try:
            res=requests.get(BASE + filename, timeout=35); res.raise_for_status()
            return pd.read_csv(io.BytesIO(res.content))
        except Exception as exc:
            raise RuntimeError(f'Download {filename} or place it next to this notebook, then rerun.') from exc

## 2. SpaceX REST API — data collection
Fetch a live snapshot of the SpaceX launch endpoint, normalize nested columns, and document the response. Historic IBM datasets are used for training because they provide curated booster-level landing labels. Live API launch records and historical course files have different schemas and should **not** be silently joined.

In [ ]:
api_url='https://api.spacexdata.com/v4/launches'
try:
    response=requests.get(api_url,timeout=30); response.raise_for_status()
    launches=pd.json_normalize(response.json())
    print('SpaceX API response:',response.status_code,'launch records:',len(launches))
    display(launches[['name','date_utc','success']].head() if {'name','date_utc','success'}.issubset(launches.columns) else launches.head())
except requests.RequestException as exc:
    print('API temporarily unavailable; historical analysis below is independent:',exc)
    launches=pd.DataFrame()

## 3. Wikipedia web scraping — collection procedure
This table extraction is a separate demonstration of HTML parsing; Wikipedia may change its markup. We count tables and preview launch-like table headings rather than incorrectly treating scraped mission-success values as booster-landing labels.

In [ ]:
wiki_url='https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'
try:
    page=requests.get(wiki_url,headers={'User-Agent':'EducationalDataScienceProject/1.0'},timeout=35)
    page.raise_for_status()
    soup=BeautifulSoup(page.text,'html.parser')
    tables=soup.select('table.wikitable')
    print('HTML tables with class wikitable:',len(tables))
    print('Example table headings:',[x.get_text(' ',strip=True) for x in tables[0].select('th')[:8]] if tables else [])
    extracted=[]
    for table in tables:
        for tr in table.select('tr'):
            values=[col.get_text(' ',strip=True) for col in tr.select('th, td')]
            if len(values)>=4: extracted.append(values[:6])
    print('Structured rows extracted:',len(extracted))
    display(pd.DataFrame(extracted[:10]))
except requests.RequestException as exc: print('Scraping unavailable:',exc)

## 4. Wrangling and label quality
Use the course-aligned launch dataset, validate the binary landing target, check missing values and duplicates, and explicitly describe the meaning of `Class`: 1 = successful first-stage landing according to the historical course labeling, 0 = otherwise. The source contains outcomes such as ocean landings, so this label is **not** the same as mission success.

In [ ]:
raw=load_csv('dataset_part_2.csv')
assert {'Class','FlightNumber','LaunchSite','Orbit','PayloadMass'}.issubset(raw.columns)
df=raw.copy()
df['Date']=pd.to_datetime(df['Date'],errors='coerce')
df['PayloadMass']=pd.to_numeric(df['PayloadMass'],errors='coerce')
df['Class']=pd.to_numeric(df['Class'],errors='coerce')
df=df[df['Class'].isin([0,1])].copy()
df['Class']=df['Class'].astype(int)
print('Shape:',df.shape,'duplicates:',df.duplicated().sum())
print('Class frequency:'); display(df['Class'].value_counts().rename(index={0:'not successful',1:'successful'}))
print('Missing values:'); display(df.isna().sum().sort_values(ascending=False).head(12))
print('Date span:',df['Date'].min(),df['Date'].max())
assert df['Class'].nunique()==2

## 5. Exploratory data analysis — SQL
SQLite demonstrates relational aggregation. Every query below is executed on the actual loaded historical rows.

In [ ]:
with sqlite3.connect(':memory:') as con:
    df.to_sql('launches',con,index=False,if_exists='replace')
    for label,q in {
      'launch site performance': "SELECT LaunchSite, COUNT(*) AS launches, ROUND(AVG(Class)*100,1) AS landing_pct FROM launches GROUP BY LaunchSite ORDER BY launches DESC",
      'orbit performance': "SELECT Orbit, COUNT(*) AS launches, ROUND(AVG(Class)*100,1) AS landing_pct FROM launches GROUP BY Orbit ORDER BY launches DESC",
      'booster reuse': "SELECT Reused, COUNT(*) AS launches, ROUND(AVG(Class)*100,1) AS landing_pct FROM launches GROUP BY Reused",
      'annual trend': "SELECT SUBSTR(Date,1,4) AS year,COUNT(*) AS launches,ROUND(AVG(Class)*100,1) AS landing_pct FROM launches GROUP BY SUBSTR(Date,1,4) ORDER BY year"
    }.items():
        print('\n'+label); display(pd.read_sql_query(q,con))

## 6. EDA with data visualization
The following charts show the relationship between launch site, orbit, payload, flight number, and landing outcomes; all charts are computed using the downloaded data.

In [ ]:
fig,axs=plt.subplots(2,2,figsize=(13,9))
sns.barplot(data=df.groupby('LaunchSite',as_index=False)['Class'].mean(),x='LaunchSite',y='Class',ax=axs[0,0])
axs[0,0].set(title='Historical landing rate by site',ylabel='Rate',xlabel='Launch site');axs[0,0].tick_params(axis='x',rotation=20)
sns.barplot(data=df.groupby('Orbit',as_index=False)['Class'].mean().sort_values('Class',ascending=False),x='Orbit',y='Class',ax=axs[0,1]);axs[0,1].tick_params(axis='x',rotation=40);axs[0,1].set(title='Historical landing rate by orbit',ylabel='Rate')
sns.scatterplot(data=df,x='FlightNumber',y='PayloadMass',hue='Class',ax=axs[1,0]);axs[1,0].set(title='Payload vs. flight number',ylabel='Payload (kg)')
annual=df.dropna(subset=['Date']).assign(Year=lambda d:d.Date.dt.year).groupby('Year')['Class'].mean()
annual.plot(marker='o',ax=axs[1,1]);axs[1,1].set(title='Historical landing rate by year',ylabel='Rate',ylim=(-.05,1.05))
plt.tight_layout();plt.show()

## 7. Folium interactive geographic map
Plot course-provided launch site coordinates; mark sites with observed success rates. Distances are geodesic estimates and are not to be interpreted as measured flight trajectories.

In [ ]:
import folium
from folium.plugins import MarkerCluster
sites=df.dropna(subset=['Latitude','Longitude']).groupby(['LaunchSite','Latitude','Longitude'],as_index=False).agg(launches=('Class','size'),success_rate=('Class','mean'))
m=folium.Map(location=[31.5,-99],zoom_start=4,tiles='CartoDB positron')
for row in sites.itertuples():
    folium.CircleMarker(location=[row.Latitude,row.Longitude],radius=8+min(row.launches,50)/5,
      popup=f'{row.LaunchSite}: {row.launches} launches, {row.success_rate:.1%} landing rate',
      color='#167d8d',fill=True,fill_opacity=.75).add_to(m)
display(m)
m.save('Falcon9_Launch_Sites_Map.html')

## 8. Plotly and Dash interactive dashboard
The Plotly figures display inline; the optional Dash application code below creates site and payload controls. Running a Dash web server is optional when submitting the notebook.

In [ ]:
import plotly.express as px
site_stats=df.groupby('LaunchSite')['Class'].agg(['mean','size']).reset_index()
fig=px.bar(site_stats,x='LaunchSite',y='mean',color='size',title='Landing success rate by launch site',labels={'mean':'Success rate','size':'Launches'})
fig.show()
fig2=px.scatter(df,x='PayloadMass',y='FlightNumber',color=df['Class'].map({0:'No landing',1:'Landing'}),hover_data=['LaunchSite','Orbit'],title='Payload and flight number vs. recorded landing result')
fig2.show()

In [ ]:
# Save a working Dash app as a separate Python source file.
dash_source = r"""import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
DATA_URL = "https://raw.githubusercontent.com/elgels/IBM-Data-Science-Capstone/main/data/dataset_part_2.csv"
data = pd.read_csv(DATA_URL)
app = Dash(__name__)
app.layout = html.Div([
    html.H2("Falcon 9 historical landing dashboard"),
    dcc.Dropdown(id="site", options=[{"label":"All Sites","value":"ALL"}] +
                 [{"label":s,"value":s} for s in sorted(data.LaunchSite.unique())], value="ALL"),
    dcc.RangeSlider(id="payload",min=0,max=16000,step=500,value=[0,16000],
                    marks={0:"0kg",8000:"8,000kg",16000:"16,000kg"}),
    dcc.Graph(id="success"), dcc.Graph(id="payloadgraph")
])
@app.callback(Output("success","figure"),Output("payloadgraph","figure"),
              Input("site","value"),Input("payload","value"))
def update(site,payload):
    s=data if site=="ALL" else data[data.LaunchSite==site]
    s=s[s.PayloadMass.between(payload[0],payload[1])]
    if s.empty: return px.scatter(title="No matching launches"),px.scatter(title="No matching launches")
    group=s.groupby("LaunchSite",as_index=False).Class.mean()
    return (px.bar(group,x="LaunchSite",y="Class",title="Landing success by site"),
            px.scatter(s,x="PayloadMass",y="FlightNumber",color="Class",title="Payload and outcome"))
if __name__ == "__main__": app.run(debug=True)
"""
with open('Falcon9_Dash_App.py','w',encoding='utf-8') as f: f.write(dash_source)
print('Saved Falcon9_Dash_App.py; start with: python Falcon9_Dash_App.py')

## 9. Predictive analysis — models and evaluation
**Leakage control:** `Outcome` and post-launch `LandingPad` are excluded, because they may disclose the target or unavailable future details. We also exclude grid fins, reuse count, and other fields not reliably known at a new-launch planning stage. Features are payload mass, orbit, launch site, and historical flight number. Random hold-out metrics are illustrative and may be optimistic under temporal drift; compare with time-ordered validation for operational use.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
feature_cols=['PayloadMass','FlightNumber','Orbit','LaunchSite']
X=df[feature_cols].copy(); y=df['Class'].copy()
num=['PayloadMass','FlightNumber']; cat=['Orbit','LaunchSite']
transformer=ColumnTransformer([
 ('num',Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
 ('cat',Pipeline([('fill',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)
])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)
models={
  'Logistic regression':(LogisticRegression(max_iter=2000),{'classifier__C':[.01,.1,1,10]}),
  'Support vector machine':(SVC(),{'classifier__kernel':['linear','rbf'],'classifier__C':[.1,1,10]}),
  'Decision tree':(DecisionTreeClassifier(random_state=SEED),{'classifier__max_depth':[2,3,5,None],'classifier__min_samples_leaf':[1,3,5]}),
  'K nearest neighbors':(KNeighborsClassifier(),{'classifier__n_neighbors':[3,5,7,9]})
}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
results=[]; fitted={}
for name,(estimator,params) in models.items():
    pipe=Pipeline([('features',transformer),('classifier',estimator)])
    search=GridSearchCV(pipe,params,scoring='accuracy',cv=cv,n_jobs=-1)
    search.fit(X_train,y_train)
    pred=search.predict(X_test)
    results.append({'model':name,'cv_accuracy':search.best_score_,'test_accuracy':accuracy_score(y_test,pred),'parameters':str(search.best_params_)})
    fitted[name]=search.best_estimator_
results_df=pd.DataFrame(results).sort_values('test_accuracy',ascending=False)
display(results_df)
print('Class majority baseline:',max(y_test.mean(),1-y_test.mean()))
best_name=results_df.iloc[0]['model'];best=fitted[best_name]
print('Best model by held-out test accuracy (descriptive only):',best_name)
print(classification_report(y_test,best.predict(X_test),zero_division=0))
ConfusionMatrixDisplay.from_estimator(best,X_test,y_test,display_labels=['No landing','Landing'],cmap='Blues');plt.title(best_name);plt.show()

## 10. Independent time-ordered robustness check
Train on earlier launches and evaluate on later launches; this guards against some random-split optimism, though historical shifts, policy changes, and sample size still limit inference.

In [ ]:
chron=df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
cut=int(len(chron)*.75)
tr,te=chron.iloc[:cut],chron.iloc[cut:]
if tr['Class'].nunique()>1 and len(te)>0:
    temporal=Pipeline([('features',transformer),('classifier',LogisticRegression(max_iter=2000))])
    temporal.fit(tr[feature_cols],tr['Class'])
    print('Earlier period:',tr.Date.min().date(),'-',tr.Date.max().date())
    print('Later period:',te.Date.min().date(),'-',te.Date.max().date())
    print('Temporal test accuracy:',round(accuracy_score(te['Class'],temporal.predict(te[feature_cols])),3))
else: print('Temporal split insufficient for classification')

## 11. Findings, interpretation and limitations
After executing the notebook, use the printed SQL aggregations, visualizations, and held-out scores as your **actual evidence**. Discuss whether performance differs by site, orbit, period, and mission payload. Do **not** claim unobserved numbers, assume model performance generalizes to current launches, or assume association establishes causation.

**Potential limitations:** small sample, historical observations through the course dataset date, unknown sampling and labeling limitations, time-varying technology, correlated launches, site-orbit confounding, noisy or imputed payload values, and evaluation variance. For production, source current records, define the landing outcome precisely, freeze feature availability at the decision point, validate temporally, monitor drift, and review costs of false positives/negatives.

**Evidence for grading:** Executed notebook prints API/sample rows, HTML table metadata, missing-data table, SQL results, four EDA plots, interactive Folium map, Plotly charts, Dash source, model comparison, classification report, confusion matrix, and temporal evaluation. Use these actual outputs in the associated presentation if submitting a data-populated slide deck.

**Data credit:** IBM Skills Network Applied Data Science Capstone historical teaching dataset; SpaceX API (`api.spacexdata.com`); Wikipedia launch table. Public course-compatible CSV mirror: github.com/elgels/IBM-Data-Science-Capstone/tree/main/data. That mirror is a community repository, not the official publisher.

## 12. Submission evidence export (RUN AFTER ALL ABOVE CELLS)
This cell **builds a new evidence-backed PDF using real executed outputs** and saves standalone chart PNGs and a data-quality summary. It will not run until the historical CSV has been successfully loaded. The exported PDF is the file to upload to the PDF field. Do not substitute the static outline for it.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.figure import Figure
from pathlib import Path
from textwrap import wrap
export_dir=Path('falcon9_submission_evidence');export_dir.mkdir(exist_ok=True)
chart_paths=[]
def evidence_plot(name, plot_func):
    fig,ax=plt.subplots(figsize=(10,5.7)); plot_func(ax)
    fig.tight_layout(); path=export_dir/(name+'.png'); fig.savefig(path,dpi=165,bbox_inches='tight');plt.close(fig)
    chart_paths.append(path)
evidence_plot('01_success_by_site',lambda ax: df.groupby('LaunchSite')['Class'].mean().sort_values().plot.barh(ax=ax,title='Landing success rate by launch site',xlabel='Success rate'))
evidence_plot('02_success_by_orbit',lambda ax: df.groupby('Orbit')['Class'].mean().sort_values().plot.barh(ax=ax,title='Landing success rate by orbit',xlabel='Success rate'))
evidence_plot('03_payload_scatter',lambda ax: sns.scatterplot(data=df,x='PayloadMass',y='FlightNumber',hue='Class',ax=ax))
evidence_plot('04_yearly_trend',lambda ax: df.assign(Year=df.Date.dt.year).groupby('Year')['Class'].mean().plot(ax=ax,marker='o',title='Historical landing rate by year',ylabel='Success rate'))
evidence_plot('05_model_comparison',lambda ax: results_df.set_index('model')[['cv_accuracy','test_accuracy']].plot.barh(ax=ax,title='Model comparison (accuracy)',xlim=(0,1)))
evidence_plot('06_confusion_matrix',lambda ax: ConfusionMatrixDisplay.from_estimator(best,X_test,y_test,display_labels=['No','Yes'],ax=ax,cmap='Blues',colorbar=False))
from reportlab.pdfgen import canvas as pdfcanvas
from reportlab.lib.utils import ImageReader
from reportlab.lib.pagesizes import landscape,letter
from reportlab.lib.colors import HexColor
output_pdf=export_dir/'Falcon9_Executed_Results_Submission.pdf'
c=pdfcanvas.Canvas(str(output_pdf),pagesize=landscape(letter));W,H=landscape(letter)
def sheet(title,lines):
    c.setFillColor(HexColor('#142947'));c.rect(0,H-80,W,80,stroke=0,fill=1)
    c.setFillColorRGB(1,1,1);c.setFont('Helvetica-Bold',22);c.drawString(42,H-52,title[:65]); c.setFillColorRGB(.1,.15,.23)
    text=c.beginText(45,H-108);text.setFont('Helvetica',11);text.setLeading(16)
    for line in lines:
        for chunk in wrap(str(line),104):text.textLine(chunk)
        text.textLine('')
    c.drawText(text);c.showPage()
sheet('Falcon 9 Landing Prediction | Executive summary',[
 'Business question: estimate historical first-stage landing outcome using characteristics known before launch.',
 'Source: IBM Skills Network historical teaching data (community-hosted course-compatible mirror).',
 f'Sample size: {len(df)}; success rate: {df.Class.mean():.1%}; date range: {df.Date.min().date()} to {df.Date.max().date()}.',
 f'Best by held-out accuracy: {best_name}; accuracy: {accuracy_score(y_test,best.predict(X_test)):.3f}.',
 'Caution: the best model was selected using the test split; compare models by cross-validation for unbiased selection.' ])
sheet('Data collection, wrangling and SQL',[
 'REST API source: api.spacexdata.com/v4/launches (independent live inspection).',
 'Web scraping source: en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches.',
 f'Historical CSV shape: {df.shape}; duplicated rows: {df.duplicated().sum()}.',
 'Missing fields: '+str(df.isna().sum().sort_values(ascending=False).head(7).to_dict()),
 'SQL: sqlite3 queries grouped by launch site, orbit, booster reuse and year; corresponding aggregations plotted on subsequent pages.' ])
for path in chart_paths:
    c.setFillColor(HexColor('#142947'));c.rect(0,H-64,W,64,stroke=0,fill=1)
    c.setFillColorRGB(1,1,1);c.setFont('Helvetica-Bold',19);c.drawString(37,H-42,path.stem.replace('_',' ').title())
    c.drawImage(ImageReader(str(path)),36,45,width=W-72,height=H-130,preserveAspectRatio=True,anchor='c')
    c.showPage()
for _,r in results_df.iterrows():
    sheet(str(r['model'])+' | evaluation',[f"Cross-validated accuracy: {r['cv_accuracy']:.3f}",f"Hold-out accuracy: {r['test_accuracy']:.3f}",f"Hyperparameters: {r['parameters']}", 'Evaluation: 75/25 stratified holdout with 5-fold tuning on the training portion.'])
sheet('Conclusion and responsible interpretation',[
 'Landing outcomes vary in the historical sample across site, orbit and launch year (see charts).',
 'The sample is historical, not a current operational data feed; the class label includes course-defined ocean outcomes.',
 'Limited sample size, chronological drift, confounding, and uncertainty about future mission conditions limit generalization.',
 'Recommendations: validate on future launches, recalibrate model probabilities, monitor drift and measure false-positive risk.',
 'Full executable project is in the companion .ipynb notebook; interactive Folium map is saved as HTML, and Dash app as Python.' ])
c.save();print('SUBMIT THIS EXECUTED PDF:',output_pdf.resolve())
print('Evidence plots:',len(chart_paths))
